# 📝 Notebook 01 — Human Preference Annotation UI

**RLHF Preference Trainer** · Step 1 of 5

This notebook launches a Gradio interface where you annotate which of two GPT-2 Medium responses is better across **4 quality dimensions**:
- 🎯 Helpfulness
- ✅ Factuality  
- 🛡️ Safety
- ✍️ Fluency

Annotations are saved to `data/preferences.csv`. Target: **~1,200 pairs**.

> **Runtime**: CPU is fine for this notebook. No GPU needed for annotation.

---

In [1]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────
!pip install -q transformers gradio pandas torch accelerate sentencepiece
print("✅ Dependencies installed")

✅ Dependencies installed


In [2]:
# ── Cell 2: Clone repo & set up src/ path ────────────────────────────────
import os, sys

# If running in Colab, clone the repo; otherwise use local path
if 'google.colab' in str(get_ipython()):
    if not os.path.exists('rlhf-preference-trainer'):
        !git clone https://github.com/sharma614/rlhf-preference-trainer.git
    os.chdir('rlhf-preference-trainer')
    print(f"📁 Working directory: {os.getcwd()}")
else:
    # Local: assume notebook is run from project root
    project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
    os.chdir(project_root)
    print(f"📁 Working directory: {os.getcwd()}")

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print("✅ Path configured")

Cloning into 'rlhf-preference-trainer'...
remote: Enumerating objects: 25, done.
remote: Counting objects: 100% (25/25), done.
remote: Compressing objects: 100% (19/19), done.
remote: Total 25 (delta 6), reused 25 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (25/25), 41.16 KiB | 1.47 MiB/s, done.
Resolving deltas: 100% (6/6), done.
📁 Working directory: /content/rlhf-preference-trainer
✅ Path configured


In [3]:
# ── Cell 3: Imports ──────────────────────────────────────────────────────
import sys, os, datetime, random
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from src.data_utils import (
    SEED_PROMPTS,
    init_preferences_csv,
    save_annotation,
    load_preferences,
    generate_prompt_pairs,
)
from src.ppo_config import MODEL_NAME, PREFERENCES_CSV

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"🖥️  Device: {device}")
print(f"🤗 Model: {MODEL_NAME}")

🖥️  Device: cpu
🤗 Model: gpt2-medium


In [4]:
# ── Cell 4: Load GPT-2 Medium ────────────────────────────────────────────
print(f"⏳ Loading {MODEL_NAME} (this may take ~1 min on first run)...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == 'cuda' else torch.float32,
)
model = model.to(device)
model.eval()

total_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"✅ {MODEL_NAME} loaded — {total_params:.1f}M parameters")

⏳ Loading gpt2-medium (this may take ~1 min on first run)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/718 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

...

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.52G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

✅ gpt2-medium loaded — 354.8M parameters


In [ ]:
# ── Cell 5: Generate initial prompt pairs ────────────────────────────────
os.makedirs('data', exist_ok=True)
init_preferences_csv(PREFERENCES_CSV)

print("⏳ Generating initial batch of prompt pairs...")
PAIRS = generate_prompt_pairs(
    model, tokenizer,
    prompts=SEED_PROMPTS,
    n_pairs=len(SEED_PROMPTS),  # All 30 seed prompts
    seed_a=42,
    seed_b=99,
)

# Add more variety: generate additional pairs with different seeds
extra_pairs = generate_prompt_pairs(
    model, tokenizer,
    prompts=SEED_PROMPTS,
    n_pairs=len(SEED_PROMPTS),
    seed_a=200,
    seed_b=777,
)
ALL_PAIRS = PAIRS + extra_pairs
random.shuffle(ALL_PAIRS)

print(f"✅ {len(ALL_PAIRS)} pairs ready for annotation")

[data_utils] Initialized new preferences CSV at data/preferences.csv
⏳ Generating initial batch of prompt pairs...
[data_utils] Generating 30 prompt pairs...


In [ ]:
# ── Cell 6: State management ─────────────────────────────────────────────
STATE = {
    'pair_idx': 0,
    'pairs': ALL_PAIRS,
    'session_count': 0,
    'annotator_id': f'annotator_{random.randint(1000, 9999)}',
}

def get_current_pair():
    if STATE['pair_idx'] >= len(STATE['pairs']):
        # Cycle through again with new generation seeds
        new_batch = generate_prompt_pairs(
            model, tokenizer, SEED_PROMPTS,
            n_pairs=30,
            seed_a=STATE['pair_idx'],
            seed_b=STATE['pair_idx'] + 500,
        )
        STATE['pairs'].extend(new_batch)
    return STATE['pairs'][STATE['pair_idx']]

def get_existing_count():
    try:
        df = pd.read_csv(PREFERENCES_CSV)
        return len(df)
    except Exception:
        return 0

print(f"🆔 Annotator ID: {STATE['annotator_id']}")
print(f"📊 Existing annotations: {get_existing_count()}")

In [ ]:
# ── Cell 7: Gradio Annotation UI ─────────────────────────────────────────
import gradio as gr

# CSS for side-by-side layout
CUSTOM_CSS = """
.response-box {
    border: 2px solid #e0e0e0;
    border-radius: 12px;
    padding: 16px;
    min-height: 200px;
    background: #fafafa;
    font-size: 14px;
    line-height: 1.6;
}
.label-a { border-color: #4A90D9 !important; }
.label-b { border-color: #E67E22 !important; }
.counter-box {
    text-align: center;
    font-size: 18px;
    font-weight: bold;
    padding: 8px;
    border-radius: 8px;
    background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
    color: white;
}
"""

def load_pair():
    pair = get_current_pair()
    count = get_existing_count()
    target = 1200
    progress = min(100, int(count / target * 100))
    counter = f"📊 {count} / {target} pairs annotated ({progress}%)"
    return (
        f"**Prompt:**\n\n> {pair['prompt']}",
        pair['response_a'],
        pair['response_b'],
        counter,
        gr.update(value=3),  # reset sliders to middle
        gr.update(value=3),
        gr.update(value=3),
        gr.update(value=3),
    )

def submit_annotation(preferred, helpfulness, factuality, safety, fluency):
    if preferred is None:
        return load_pair() + ("⚠️ Please select a preference before submitting!",)

    pair = get_current_pair()
    row = {
        'prompt': pair['prompt'],
        'response_a': pair['response_a'],
        'response_b': pair['response_b'],
        'preferred': preferred,
        'helpfulness_score': helpfulness,
        'factuality_score': factuality,
        'safety_score': safety,
        'fluency_score': fluency,
        'annotator_id': STATE['annotator_id'],
        'timestamp': datetime.datetime.now().isoformat(),
    }
    save_annotation(row, PREFERENCES_CSV)
    STATE['pair_idx'] += 1
    STATE['session_count'] += 1

    return load_pair() + (f"✅ Saved! Session: {STATE['session_count']} annotations",)

def skip_pair():
    STATE['pair_idx'] += 1
    return load_pair() + ("⏭️ Skipped.",)

# ── Build UI ──────────────────────────────────────────────────────────────
with gr.Blocks(css=CUSTOM_CSS, title="RLHF Annotation UI", theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # 🧠 RLHF Preference Annotation Interface
    **Instructions:** Read the prompt, then rate both responses across 4 dimensions.
    Select which response you **prefer overall**, then click **Submit**.
    """)

    counter_display = gr.Markdown(value="📊 Loading...", elem_id="counter")
    prompt_display = gr.Markdown(label="Prompt", value="Loading...")

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 🔵 Response A")
            resp_a = gr.Textbox(
                label="Response A",
                lines=10,
                interactive=False,
                elem_classes=["response-box", "label-a"]
            )
        with gr.Column(scale=1):
            gr.Markdown("### 🟠 Response B")
            resp_b = gr.Textbox(
                label="Response B",
                lines=10,
                interactive=False,
                elem_classes=["response-box", "label-b"]
            )

    gr.Markdown("---")
    gr.Markdown("### 📏 Rate the **preferred** response across 4 dimensions (1=Poor, 5=Excellent)")

    with gr.Row():
        helpfulness_sl = gr.Slider(1, 5, value=3, step=1, label="🎯 Helpfulness")
        factuality_sl  = gr.Slider(1, 5, value=3, step=1, label="✅ Factuality")
    with gr.Row():
        safety_sl  = gr.Slider(1, 5, value=3, step=1, label="🛡️ Safety")
        fluency_sl = gr.Slider(1, 5, value=3, step=1, label="✍️ Fluency")

    gr.Markdown("### 🏆 Overall Preference")
    preference = gr.Radio(
        choices=["A", "B", "Tie"],
        label="Which response do you prefer?",
        value=None,
    )

    status_msg = gr.Markdown(value="")

    with gr.Row():
        submit_btn = gr.Button("✅ Submit & Next", variant="primary", scale=3)
        skip_btn   = gr.Button("⏭️ Skip", variant="secondary", scale=1)

    # Slider outputs list (needed for reset)
    slider_outputs = [helpfulness_sl, factuality_sl, safety_sl, fluency_sl]
    all_outputs = [prompt_display, resp_a, resp_b, counter_display] + slider_outputs

    submit_btn.click(
        fn=submit_annotation,
        inputs=[preference, helpfulness_sl, factuality_sl, safety_sl, fluency_sl],
        outputs=all_outputs + [status_msg],
    )
    skip_btn.click(
        fn=skip_pair,
        inputs=[],
        outputs=all_outputs + [status_msg],
    )

    # Load first pair on startup
    demo.load(fn=load_pair, outputs=all_outputs)

print("✅ Gradio app built successfully")

In [ ]:
# ── Cell 8: Launch annotation UI ─────────────────────────────────────────
# Set share=True for a public URL (useful in Colab)
demo.launch(share=True, debug=False)

In [ ]:
# ── Cell 9: Annotation statistics ────────────────────────────────────────
# Run this cell AFTER annotating to see current statistics

import pandas as pd
from src.ppo_config import PREFERENCES_CSV

try:
    df = pd.read_csv(PREFERENCES_CSV)
    print(f"\n{'='*50}")
    print(f"  ANNOTATION STATISTICS")
    print(f"{'='*50}")
    print(f"  Total annotations : {len(df)}")
    print(f"  Unique annotators : {df['annotator_id'].nunique()}")
    print(f"  Preference dist   :")
    for pref, cnt in df['preferred'].value_counts().items():
        pct = 100 * cnt / len(df)
        print(f"    {pref:5s}: {cnt:4d} ({pct:.1f}%)")
    print(f"\n  Mean Scores (on preferred responses):")
    for dim in ['helpfulness_score', 'factuality_score', 'safety_score', 'fluency_score']:
        mean_val = df[dim].mean()
        print(f"    {dim.replace('_score','').title():15s}: {mean_val:.2f}/5.00")
    print(f"{'='*50}")

    print("\n  Sample of annotated pairs:")
    display(df[['prompt', 'preferred', 'helpfulness_score', 'fluency_score']].head(5))
except FileNotFoundError:
    print("No annotations yet. Run the Gradio UI and annotate some pairs first!")

In [ ]:
# ── Cell 10: Smoke test (verify CSV structure) ────────────────────────────
import os
from src.ppo_config import PREFERENCES_CSV

assert os.path.exists(PREFERENCES_CSV), f"ERROR: {PREFERENCES_CSV} not found!"
df_check = pd.read_csv(PREFERENCES_CSV)
required_cols = ['prompt', 'response_a', 'response_b', 'preferred',
                 'helpfulness_score', 'factuality_score', 'safety_score', 'fluency_score']
for col in required_cols:
    assert col in df_check.columns, f"ERROR: Missing column '{col}'"
print(f"✅ Smoke test PASSED — CSV has {len(df_check)} rows and all required columns.")
print(f"   Next step: Run notebook 02_reward_model_training.ipynb")